# Full cohort preprocessing run (163 subjects)

**Objective**: run `preprocess_subject` across the full TDBRAIN-filtered cohort, producing
**two variants** in the same run: HEOG correction off (default, current best-validated state)
and HEOG correction on (using the same baseline-drift-removed, duration-bounded segments).
Running both now, rather than deferring, converts the open HEOG-confidence question into two
ready-to-compare datasets for the modelling-stage sensitivity check, at the cost of
approximately doubling this run's compute time.

**Inputs**:
- `src/preprocessing.py` - full pipeline, validated on pilot subject + 6-subject stratified
  batch (see `notebooks/03_batch_test.ipynb`, `docs/preprocessing_notes.md`)
- `data/cohort_filtered_n163.xlsx` - full filtered cohort subject list

**Design choices**:
- **Resumable**: checks for existing output before reprocessing a subject/variant, so an
  interruption doesn't require starting over.
- **Two output locations**: `data/derivatives_heog_off/` and `data/derivatives_heog_on/`,
  keeping both variants' saved epochs separate and clearly labelled.
- **Results logged incrementally to disk** as a flat CSV (one row per subject/condition/variant,
  all QC fields included, including a `heog_variant` column).
- **Defaults used throughout otherwise**: no per-subject parameter tuning beyond the
  `apply_heog_correction` flag itself.

**Known open items going into this run** (see `docs/preprocessing_notes.md` for full detail):
- HEOG correction confidence is improved (baseline-drift removal, validated window size) but
  not fully resolved - roughly half of detected segments still show step/trend character
  rather than isolated events. This run produces the data needed to test empirically whether
  correction on vs. off matters for downstream classification.
- `autoreject` consensus/n_interpolate instability is expected to recur at some rate; QC flags
  will be reviewed after this run completes.
- Orchestrator's per-condition error handling has not yet been triggered by a real failure -
  this run is the first at a scale where that's plausible.
- Known code inconsistency (documented in `gratton_regression`'s context): if HEOG correction
  is applied, it regresses against non-detrended HEOG data, not the detrended version used for
  detection. This means the "HEOG on" variant produced here inherits that specific limitation -
  worth remembering when interpreting its results, not a reason to skip generating it.

In [2]:
# imports and setup
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
import importlib
import src.preprocessing
importlib.reload(src.preprocessing)
from src.preprocessing import *

In [ ]:
#Cohort loading 
data_dir = find_repo_root() / "data"

cohort_df = pd.read_excel(data_dir / "cohort_filtered_n163.xlsx")
all_subject_ids = cohort_df['TDBRAIN_ID'].tolist()
print(f"{len(all_subject_ids)} subjects in cohort")

163 subjects in cohort


In [6]:
#Time check for one participant 
import time

t0 = time.time()
_ = preprocess_subject(all_subject_ids[0], data_dir, apply_heog_correction=False, derivatives_dirname='derivatives_heog_off')
_ = preprocess_subject(all_subject_ids[0], data_dir, apply_heog_correction=True, derivatives_dirname='derivatives_heog_on')
elapsed = time.time() - t0
print(f"One subject, both variants, both conditions: {elapsed:.0f}s")
print(f"Estimated total for {len(all_subject_ids)} subjects: {elapsed * len(all_subject_ids) / 60:.0f} min")

Extracting BDF parameters from /Users/romyweinstock/eeg-rtms-response-prediction/data/TDBRAIN_Dataset_V3_1/sub-87999321/ses-1/eeg/sub-87999321_ses-1_task-restEC_eeg.bdf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 60499  =      0.000 ...   120.998 secs...
Creating RawArray with float64 data, n_channels=2, n_times=60500
    Range : 0 ... 60499 =      0.000 ...   120.998 secs
Ready.
Not setting metadata
24 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 24 events and 2500 original time points ...
0 bad epochs dropped
Running autoreject on ch_type=eeg


  0%|          | Creating augmented epochs : 0/26 [00:00<?,       ?it/s]

  0%|          | Computing thresholds ... : 0/26 [00:00<?,       ?it/s]

  0%|          | Repairing epochs : 0/24 [00:00<?,       ?it/s]

  0%|          | n_interp : 0/3 [00:00<?,       ?it/s]

...

/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


  0%|          | Fold : 0/10 [00:00<?,       ?it/s]





Estimated consensus=0.20 and n_interpolate=4


  0%|          | Repairing epochs : 0/24 [00:00<?,       ?it/s]

Dropped 2 epochs: 22, 23
One subject, both variants, both conditions: 40s
Estimated total for 163 subjects: 109 min


In [7]:
#Before running the full cohort confirm both variants produced different output
import os
off_files = list((data_dir / 'derivatives_heog_off' / all_subject_ids[0]).glob('*.fif'))
on_files = list((data_dir / 'derivatives_heog_on' / all_subject_ids[0]).glob('*.fif'))
print("off:", off_files)
print("on:", on_files)

off: [PosixPath('/Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-87999321/sub-87999321_restEO-epo.fif'), PosixPath('/Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-87999321/sub-87999321_restEC-epo.fif')]
on: [PosixPath('/Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_on/sub-87999321/sub-87999321_restEO-epo.fif'), PosixPath('/Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_on/sub-87999321/sub-87999321_restEC-epo.fif')]


In [8]:
# Run the preprocessing on all subject twice with the two variants
import time

variants = [
    {'label': 'heog_off', 'apply_heog_correction': False, 'derivatives_dirname': 'derivatives_heog_off'},
    {'label': 'heog_on',  'apply_heog_correction': True,  'derivatives_dirname': 'derivatives_heog_on'},
]

results_log_path = data_dir / "batch_results_log_full_cohort.csv"
results_rows = []

if results_log_path.exists():
    existing_log = pd.read_csv(results_log_path)
    already_done = set(zip(existing_log['subject_id'], existing_log['condition'], existing_log['heog_variant']))
    results_rows = existing_log.to_dict('records')
    print(f"Resuming: {len(already_done)} subject/condition/variant combinations already logged")
else:
    already_done = set()
    print("No existing log found - starting fresh")

start_time = time.time()
for i, sid in enumerate(all_subject_ids):
    for variant in variants:
        already_has_both_conditions = (
            (sid, 'restEC', variant['label']) in already_done and
            (sid, 'restEO', variant['label']) in already_done
        )
        if already_has_both_conditions:
            print(f"[{i+1}/{len(all_subject_ids)}] {sid} ({variant['label']}): already done, skipping")
            continue

        result = preprocess_subject(
            sid, data_dir,
            apply_heog_correction=variant['apply_heog_correction'],
            derivatives_dirname=variant['derivatives_dirname']
        )

        for condition, r in result.items():
            if (sid, condition, variant['label']) in already_done:
                continue
            row = {
                'subject_id': sid, 'condition': condition, 'heog_variant': variant['label'],
                **{k: v for k, v in r.items() if k != 'emg_audit_df'}
            }
            results_rows.append(row)

    pd.DataFrame(results_rows).to_csv(results_log_path, index=False)
    elapsed = time.time() - start_time
    print(f"[{i+1}/{len(all_subject_ids)}] {sid} done ({elapsed/60:.1f} min elapsed)")

print(f"\nComplete. Total time: {(time.time()-start_time)/60:.1f} min")

No existing log found - starting fresh
Extracting BDF parameters from /Users/romyweinstock/eeg-rtms-response-prediction/data/TDBRAIN_Dataset_V3_1/sub-87999321/ses-1/eeg/sub-87999321_ses-1_task-restEC_eeg.bdf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 60499  =      0.000 ...   120.998 secs...
Creating RawArray with float64 data, n_channels=2, n_times=60500
    Range : 0 ... 60499 =      0.000 ...   120.998 secs
Ready.
Not setting metadata
24 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 24 events and 2500 original time points ...
0 bad epochs dropped
Running autoreject on ch_type=eeg


  0%|          | Creating augmented epochs : 0/26 [00:00<?,       ?it/s]

  0%|          | Computing thresholds ... : 0/26 [00:00<?,       ?it/s]

  0%|          | Repairing epochs : 0/24 [00:00<?,       ?it/s]

  0%|          | n_interp : 0/3 [00:00<?,       ?it/s]

...

/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/miniconda3/envs/eeg-rtms/lib/python3.11/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


  0%|          | Fold : 0/10 [00:00<?,       ?it/s]





Estimated consensus=0.30 and n_interpolate=1


  0%|          | Repairing epochs : 0/24 [00:00<?,       ?it/s]

Dropped 1 epoch: 23
[160/163] sub-88025061 done (108.2 min elapsed)
[161/163] sub-88026321 done (108.2 min elapsed)
[162/163] sub-88022133 done (108.2 min elapsed)
[163/163] sub-88041397 done (108.2 min elapsed)

Complete. Total time: 108.2 min


In [9]:
log = pd.read_csv(data_dir / "batch_results_log_full_cohort.csv")
print(f"Total rows: {len(log)} (expect 163 subjects × 2 conditions × 2 variants = 652)")
print(f"Unique subjects: {log['subject_id'].nunique()} (expect 163)")
print(f"Status counts:\n{log['status'].value_counts()}")

Total rows: 652 (expect 163 subjects × 2 conditions × 2 variants = 652)
Unique subjects: 163 (expect 163)
Status counts:
status
ok       640
error     12
Name: count, dtype: int64


In [10]:
errors = log[log['status'] == 'error']
print(f"{len(errors)} error rows")
print(errors[['subject_id', 'condition', 'heog_variant']].to_string(index=False))
print()
print("Unique error messages:")
print(errors['error'].value_counts())

12 error rows
  subject_id condition heog_variant
sub-88026321    restEC     heog_off
sub-88026321    restEO     heog_off
sub-88026321    restEC      heog_on
sub-88026321    restEO      heog_on
sub-88022133    restEC     heog_off
sub-88022133    restEO     heog_off
sub-88022133    restEC      heog_on
sub-88022133    restEO      heog_on
sub-88041397    restEC     heog_off
sub-88041397    restEO     heog_off
sub-88041397    restEC      heog_on
sub-88041397    restEO      heog_on

Unique error messages:
error
The file /Users/romyweinstock/eeg-rtms-response-prediction/data/TDBRAIN_Dataset_V3_1/sub-88026321/ses-1/eeg/sub-88026321_ses-1_task-restEC_eeg.bdf does not exist.    2
The file /Users/romyweinstock/eeg-rtms-response-prediction/data/TDBRAIN_Dataset_V3_1/sub-88026321/ses-1/eeg/sub-88026321_ses-1_task-restEO_eeg.bdf does not exist.    2
The file /Users/romyweinstock/eeg-rtms-response-prediction/data/TDBRAIN_Dataset_V3_1/sub-88022133/ses-1/eeg/sub-88022133_ses-1_task-restEC_eeg.bdf does 

In [11]:
import os
for sid in ['sub-88026321', 'sub-88022133', 'sub-88041397']:
    subj_dir = data_dir / "TDBRAIN_Dataset_V3_1" / sid / "ses-1" / "eeg"
    print(sid, "-> directory exists:", subj_dir.exists())
    if subj_dir.exists():
        print("  contents:", os.listdir(subj_dir))

sub-88026321 -> directory exists: False
sub-88022133 -> directory exists: False
sub-88041397 -> directory exists: False


In [12]:
log_valid = log[log['status'] == 'ok'].copy()
print(f"Usable subjects: {log_valid['subject_id'].nunique()} of 163")

Usable subjects: 160 of 163


In [14]:
log_valid = log[log['status'] == 'ok'].copy()
print(f"Usable subjects: {log_valid['subject_id'].nunique()} of 163")

# Autoreject QC flag rate
print("\nExtreme autoreject flag rate:")
print(log_valid.groupby('heog_variant')['autoreject_extreme'].mean())

# Epoch retention
log_valid['pct_retained'] = log_valid['n_epochs_after'] / log_valid['n_epochs_before']
print("\nEpoch retention (median, min):")
print(log_valid.groupby('heog_variant')['pct_retained'].agg(['median', 'min']))

# HEOG segment counts across the full cohort (same detection regardless of variant)
heog_summary = log_valid[log_valid['heog_variant']=='heog_off'][['heog_n_candidates', 'heog_n_valid']]
print("\nHEOG candidate/valid counts across cohort:")
print(heog_summary.describe())

Usable subjects: 160 of 163

Extreme autoreject flag rate:
heog_variant
heog_off     0.38125
heog_on     0.371875
Name: autoreject_extreme, dtype: object

Epoch retention (median, min):
                median       min
heog_variant                    
heog_off      0.958333  0.416667
heog_on       0.958333  0.416667

HEOG candidate/valid counts across cohort:
       heog_n_candidates  heog_n_valid
count         320.000000    320.000000
mean           73.590625     14.275000
std            23.709274      9.428796
min             4.000000      0.000000
25%            57.000000      6.000000
50%            72.000000     12.000000
75%            90.000000     21.000000
max           155.000000     53.000000


In [15]:
worst = log_valid.nsmallest(5, 'pct_retained')[['subject_id', 'condition', 'heog_variant', 'n_epochs_before', 'n_epochs_after', 'pct_retained', 'autoreject_extreme']]
print(worst)

       subject_id condition heog_variant  n_epochs_before  n_epochs_after  \
325  sub-88068885    restEO     heog_off             24.0            10.0   
327  sub-88068885    restEO      heog_on             24.0            10.0   
205  sub-88072581    restEO     heog_off             24.0            16.0   
207  sub-88072581    restEO      heog_on             24.0            16.0   
309  sub-88066773    restEO     heog_off             24.0            16.0   

     pct_retained autoreject_extreme  
325      0.416667              False  
327      0.416667              False  
205      0.666667              False  
207      0.666667              False  
309      0.666667              False  
